# 🧃 بدّلها لطفلي — Notebook 2 (Analytics): التحليل ونظام البدائل

هذي النسخة مركّزة على **Data Analysis** و**Data Engineering**، بدون Machine Learning. نسخة الـ ML محفوظة لحالها (`02_ml_nutriscore_swaps_v2.ipynb`) وتقدرين تضيفينها بعدين.

**وش يسوي هالـ Notebook:**
1. يقرأ منتجات الأطفال النظيفة من طبقة **Gold** (ناتج Notebook 1).
2. يحلل البيانات بـ **Spark SQL**: توزيع التقييم الغذائي، الماركات، أكثر المنتجات سكراً، والسكر في علبة العصير.
3. يبني **نظام بدائل** يقترح لكل منتج بديل أصح من نفس النوع (قواعد وتشابه، مو ML).
4. يتأكد من سلامة البيانات وقواعد البدائل باختبارات تلقائية، وبعدها يحفظ النتائج.

**التقييم الغذائي:** نستخدم **Nutri-Score الرسمي بس**. المنتج اللي ما له تقييم رسمي يبقى "بدون تقييم"، وما نخمّن له.

**التشغيل:** شغّلي Notebook 1 أول مرة، وبعدها هذا. التشغيل كامل تقريباً دقيقتين.

## 1) التجهيز: ربط Drive وتشغيل Spark
- نربط Google Drive عشان نقرأ ملفات Notebook 1 ونحفظ النتائج.
- نثبّت PySpark ونشغّل جلسة Spark بذاكرة 4g و8 أجزاء (البيانات هنا صغيرة، تقريباً ألف منتج).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/kids_food_swap'
GOLD     = f'{PROJECT_DIR}/gold'
REPORTS  = f'{PROJECT_DIR}/reports'
os.makedirs(REPORTS, exist_ok=True)
assert os.path.exists(f'{GOLD}/kids_products.parquet'), 'شغّلي Notebook 1 أول'
print('✅ جاهز')

!pip install -q pyspark

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
import pandas as pd
import numpy as np

spark = (SparkSession.builder
         .appName('KidsFoodSwap-Analytics')
         .config('spark.driver.memory', '4g')
         .config('spark.sql.shuffle.partitions', '8')
         .getOrCreate())
spark

## 2) قراءة البيانات وتجهيزها للتحليل
- نقرأ ملف Gold من Notebook 1، ونختار الأعمدة اللي نحتاجها بأسماء قصيرة بدون شرطات، عشان نكتب SQL بسهولة.
- `salt_clean` هو الملح بعد استبعاد القيم الشاذة (أكثر من 5 جرام لكل 100 جرام).
- **حالة التقييم** (`grade_status`):
  - `official`: له Nutri-Score رسمي.
  - `no official grade`: ما له تقييم. ما نخمّن له، نخليه بدون تقييم.
  - `not applicable`: أغذية الرضع، لأن Nutri-Score مو مصمم لها.
- `createOrReplaceTempView` تسجل الجدول باسم، عشان نستعلم عنه بـ SQL.

In [ ]:
GRADES = ['a', 'b', 'c', 'd', 'e']

kids = spark.read.parquet(f'{GOLD}/kids_products.parquet')
products = (kids.select(
        'code', 'product_name', 'brands', 'categories_tags', 'product_group', 'audience',
        F.col('sugars_100g').alias('sugars'),
        F.col('sugar_tsp_100g').alias('sugar_tsp'),
        F.col('salt_clean').alias('salt'),
        F.col('energy-kcal_100g').alias('energy_kcal'),
        F.col('fat_100g').alias('fat'),
        F.col('saturated-fat_100g').alias('sat_fat'),
        F.col('proteins_100g').alias('proteins'),
        F.lower(F.trim('nutriscore_grade')).alias('nutriscore_grade'))
    .withColumn('grade', F.when((F.col('product_group') != 'Baby food') & F.col('nutriscore_grade').isin(GRADES),
                                F.col('nutriscore_grade')))
    .withColumn('grade_status', F.when(F.col('product_group') == 'Baby food', 'not applicable')
                                 .when(F.col('grade').isNull(), 'no official grade')
                                 .otherwise('official'))
    .drop('nutriscore_grade')
    .cache())

products.createOrReplaceTempView('products')
print('عدد المنتجات:', products.count())
products.groupBy('grade_status').count().show()

## 3) التحليل بـ Spark SQL 🔍
نفس التحليل ممكن ينكتب بـ PySpark أو بـ SQL، والاثنين يتحولون لنفس خطة التنفيذ. هنا نستخدم **SQL** لأنه أهم مهارة في وظائف Data Analyst.

دالة `run_sql` تشغّل الاستعلام، وتعرض النتيجة، وتحفظها CSV في `reports` عشان نستخدمها في README.

In [ ]:
def run_sql(name, query):
    result = spark.sql(query).toPandas()     # النتائج صغيرة (ملخصات)، فـ toPandas آمنة
    result.to_csv(f'{REPORTS}/analytics_{name}.csv', index=False, encoding='utf-8-sig')
    display(result)
    return result

### أ) توزيع التقييم الغذائي لكل فئة
- `SUM(CASE WHEN ... THEN 1 ELSE 0 END)` يحسب كم منتج في كل تقييم. هذا اسمه **Pivot يدوي** بـ SQL.
- `pct_d_or_e`: نسبة المنتجات اللي تقييمها D أو E (الأسوأ). `AVG` على عمود فيه 1 و0 = نسبة الـ 1.

In [ ]:
grade_dist = run_sql('grade_distribution', """
SELECT product_group,
       COUNT(*)                                   AS graded_products,
       SUM(CASE WHEN grade = 'a' THEN 1 ELSE 0 END) AS a,
       SUM(CASE WHEN grade = 'b' THEN 1 ELSE 0 END) AS b,
       SUM(CASE WHEN grade = 'c' THEN 1 ELSE 0 END) AS c,
       SUM(CASE WHEN grade = 'd' THEN 1 ELSE 0 END) AS d,
       SUM(CASE WHEN grade = 'e' THEN 1 ELSE 0 END) AS e,
       ROUND(100 * AVG(CASE WHEN grade IN ('d', 'e') THEN 1.0 ELSE 0 END), 1) AS pct_d_or_e
FROM products
WHERE grade IS NOT NULL
GROUP BY product_group
ORDER BY pct_d_or_e DESC
""")

### ب) الماركات الأكثر حضوراً
- `SPLIT(brands, ',')[0]` تاخذ أول ماركة، لأن بعض المنتجات مكتوب فيها أكثر من ماركة.
- `INITCAP(LOWER(...))` توحّد الكتابة، عشان `ALMARAI` و`almarai` يصيرون ماركة وحدة.
- `HAVING COUNT(*) >= 5`: نعرض الماركات اللي عندها 5 منتجات أو أكثر بس، عشان النسب تكون لها معنى.
- `CASE WHEN grade IS NULL THEN NULL ...`: المنتجات بدون تقييم ما تدخل في حساب النسبة (AVG يتجاهل NULL).

In [ ]:
brands = run_sql('top_brands', """
SELECT INITCAP(LOWER(TRIM(SPLIT(brands, ',')[0])))            AS brand,
       COUNT(*)                                              AS products,
       ROUND(PERCENTILE_APPROX(sugars, 0.5), 1)              AS median_sugar_g,
       ROUND(100 * AVG(CASE WHEN grade IS NULL THEN NULL
                            WHEN grade IN ('d', 'e') THEN 1.0 ELSE 0 END), 1) AS pct_d_or_e
FROM products
WHERE brands IS NOT NULL AND TRIM(brands) <> '' AND product_group <> 'Baby food'
GROUP BY 1
HAVING COUNT(*) >= 5
ORDER BY products DESC
LIMIT 15
""")

### ج) أكثر 3 منتجات سكراً في كل فئة (Window Function)
`ROW_NUMBER() OVER (PARTITION BY ... ORDER BY ...)` يرقّم المنتجات داخل كل فئة من الأعلى سكراً للأقل، وبعدها ناخذ أول 3. من أكثر الأسئلة تكراراً في مقابلات SQL.

In [ ]:
top_sugar = run_sql('top_sugar_per_group', """
SELECT product_group, rn AS rank, product_name, brands, sugars, sugar_tsp
FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY product_group ORDER BY sugars DESC, product_name) AS rn
    FROM products
    WHERE product_group <> 'Baby food'
) ranked
WHERE rn <= 3
ORDER BY product_group, rn
""")

### د) كم ملعقة سكر في علبة العصير؟ 🧃
قيم السكر مكتوبة **لكل 100 مل**، والأم تشتري علبة كاملة. فنحسب السكر في **علبة 250 مل** (الحجم الشائع)، ونقارنه بـ **الحد اليومي الموصى به للأطفال: أقل من 25 جرام سكر مضاف** (جمعية القلب الأمريكية).

**تنبيه صادق:** الملصق يكتب السكر **الكلي**، مو المضاف بس. في العصير الطبيعي 100% السكر من الفاكهة نفسها، لكن منظمة الصحة العالمية تعتبر سكر العصير "سكر حر" مثل المضاف. فالمقارنة تقريبية، ونكتبها بهالشكل في الداشبورد.

In [ ]:
drinks = run_sql('drinks_per_250ml', """
SELECT product_group,
       COUNT(*)                                                   AS products,
       ROUND(PERCENTILE_APPROX(sugars, 0.5), 1)                   AS median_sugar_per_100ml,
       ROUND(PERCENTILE_APPROX(sugars, 0.5) * 2.5, 1)             AS median_sugar_g_per_250ml,
       ROUND(PERCENTILE_APPROX(sugars, 0.5) * 2.5 / 4, 1)         AS median_tsp_per_250ml,
       ROUND(100 * AVG(CASE WHEN sugars * 2.5 >= 25 THEN 1.0 ELSE 0 END), 1) AS pct_box_reaches_daily_limit
FROM products
WHERE product_group IN ('Juices', 'Soft drinks', 'Flavoured milk')
GROUP BY product_group
ORDER BY median_sugar_per_100ml DESC
""")

## 4) نظام اقتراح البديل الصحي 🔄
**شروط البديل:** من نفس الفئة، وتقييمه الرسمي أفضل، وسكره أقل بـ 30% على الأقل، ومو نفس المنتج باسم ثاني.

**كيف نرتّب البدائل:**
| الجزء | وش يسوي |
| --- | --- |
| **تشابه الوسوم (Jaccard)** | الوسوم المشتركة ÷ كل الوسوم. مصاص ومصاص ثاني يشتركون في `lollipops` |
| **الوسوم المميزة فقط** | نشيل الوسوم الموجودة في أكثر من 60% من منتجات الفئة (مثل `sweet-snacks`)، لأنها ما تفرق |
| **الترتيب** | الأشبه في النوع أول، وعند التعادل الأكثر توفيراً للسكر |
| **حد التكرار** | نفس البديل ما ينقترح أكثر من 5 مرات (بالاسم، لأن المنتج ممكن له أكثر من باركود) |

هذا **مو Machine Learning**: ما فيه نموذج يتعلم من بيانات. هي قواعد ومقاييس تشابه نكتبها بأنفسنا.

نقارن نسختين: **v1** (أكبر توفير سكر بس، بدون حد تكرار) و**v2** (وسوم + تنويع)، عشان نثبت بالأرقام إن v2 أحسن. البيانات صغيرة، فنستخدم Pandas.

In [ ]:
from collections import Counter

GRADE_SCORE = {'a': 5, 'b': 4, 'c': 3, 'd': 2, 'e': 1}

pdf = (products.filter(F.col('grade_status') == 'official')
               .select('code', 'product_name', 'brands', 'product_group', 'categories_tags',
                       F.col('grade').alias('final_grade'), 'sugars')
               .toPandas())
pdf['health_score'] = pdf['final_grade'].map(GRADE_SCORE)
pdf['name_key'] = pdf['product_name'].fillna('').str.lower().str.strip()

# الوسوم المميزة لكل منتج (بدون الوسوم العامة في فئته)
tag_map = {}
for grp, g in pdf.groupby('product_group'):
    sets = [set(t.strip() for t in str(x).lower().split(',') if t.strip()) if pd.notna(x) else set()
            for x in g['categories_tags']]
    freq = Counter(t for s in sets for t in s)
    common = {t for t, n in freq.items() if n / len(g) > 0.6}
    for idx, s in zip(g.index, sets):
        tag_map[idx] = s - common
pdf['tagset'] = pd.Series(tag_map)

def jaccard(a, b):
    u = a | b
    return len(a & b) / len(u) if u else 0.3   # منتجين بدون وسوم مميزة: تشابه محايد

def recommend(df, use_tags, max_uses, min_tag_sim, top_k=3):
    uses, rows = Counter(), []
    for group, g in df.groupby('product_group'):
        for _, p in g.sort_values('sugars', ascending=False).iterrows():   # الأكثر سكراً ياخذ الأولوية
            # 1) الشروط: تقييم أفضل، سكر أقل بـ 30% على الأقل، مو نفس المنتج
            cand = g[(g['health_score'] > p['health_score']) &
                     (g['sugars'] <= p['sugars'] * 0.7) &
                     (g['name_key'] != p['name_key'])].copy()
            if cand.empty:
                continue
            # 2) التشابه في النوع، وكم سكر نوفّر
            cand['tag_sim'] = cand['tagset'].apply(lambda s: jaccard(s, p['tagset']))
            cand['sugar_saved'] = p['sugars'] - cand['sugars']
            # 3) حد التشابه وحد التكرار
            cand = cand[(cand['tag_sim'] >= min_tag_sim) & (cand['name_key'].map(lambda k: uses[k] < max_uses))]
            # 4) الترتيب: الأشبه أول، وعند التعادل الأكثر توفيراً للسكر
            order = ['tag_sim', 'sugar_saved'] if use_tags else ['sugar_saved']
            cand = cand.sort_values(order, ascending=False).drop_duplicates('name_key')
            for rank, (_, a) in enumerate(cand.head(top_k).iterrows(), start=1):
                uses[a['name_key']] += 1   # نعد بالاسم: نفس المنتج ممكن له أكثر من باركود
                rows.append({
                    'product_group': group, 'rank': rank,
                    'code': p['code'], 'product_name': p['product_name'], 'brands': p['brands'],
                    'grade': p['final_grade'].upper(), 'sugar_g': p['sugars'],
                    'alt_code': a['code'], 'alt_name': a['product_name'], 'alt_brands': a['brands'],
                    'alt_grade': a['final_grade'].upper(), 'alt_sugar_g': a['sugars'],
                    'sugar_saved_tsp': round(a['sugar_saved'] / 4, 1),
                    'tag_similarity': round(a['tag_sim'], 2),
                })
    return pd.DataFrame(rows)

def rec_metrics(sw, label):
    top1 = sw[sw['rank'] == 1]
    reuse = sw['alt_name'].str.lower().str.strip().value_counts()
    return {'version': label,
            'products_with_swap': sw['code'].nunique(),
            'coverage_pct': round(sw['code'].nunique() / len(pdf) * 100, 1),
            'unique_alternatives': sw['alt_code'].nunique(),
            'max_times_same_alt': int(reuse.max()),
            'avg_tag_similarity_top1': round(top1['tag_similarity'].mean(), 2),
            'avg_sugar_saved_tsp_top1': round(top1['sugar_saved_tsp'].mean(), 1)}

swaps_v1 = recommend(pdf, use_tags=False, max_uses=10**9, min_tag_sim=0.0)   # v1: أكبر توفير سكر بس
swaps = recommend(pdf, use_tags=True, max_uses=5, min_tag_sim=0.2)           # v2: وسوم + تنويع

rec_cmp = pd.DataFrame([rec_metrics(swaps_v1, 'v1: sugar only'), rec_metrics(swaps, 'v2: tags + diversity')])
rec_cmp.to_csv(f'{REPORTS}/recommender_metrics.csv', index=False, encoding='utf-8-sig')
display(rec_cmp)
print('أكثر بديل يتكرر في v1:'); display(swaps_v1['alt_name'].value_counts().head(5))
print('أكثر بديل يتكرر في v2:'); display(swaps['alt_name'].value_counts().head(5))
swaps[swaps['rank'] == 1].sort_values('sugar_saved_tsp', ascending=False).head(15)

## 5) الاختبارات والحفظ 🧪💾
قبل ما نحفظ أي شي، نتأكد إن البيانات سليمة وإن كل بديل يطبّق القواعد اللي حطيناها. لو فشل أي اختبار يوقف الكود وما يحفظ شي، عشان ما نحفظ نتائج غلط.

**وش نختبر:**
- **سلامة البيانات:** ما فيه باركود مكرر، التقييمات كلها من A إلى E بس، أغذية الرضع بدون تقييم، وما فيه سكر بالسالب.
- **قواعد البدائل:** كل بديل من نفس الفئة، تقييمه أفضل، سكره أقل بـ 30% على الأقل، ما يتكرر أكثر من 5 مرات، وتشابهه 0.2 أو أكثر.

**الحفظ** (يصير بس إذا كل الاختبارات نجحت):
- `gold/swap_recommendations.csv`: البدائل المقترحة.
- `reports/analytics_*.csv`: نتائج تحليلات SQL (انحفظت وقت التحليل).

`utf-8-sig` عشان العربي يطلع صح لما تفتحين الملف في Excel.

In [ ]:
results = []
def check(name, passed):
    results.append((name, passed))
    print('✅' if passed else '❌', name)

fp = products.toPandas()   # البيانات صغيرة (حوالي ألف منتج)، فنفحصها بـ Pandas

# سلامة البيانات
check('ما فيه باركود مكرر', fp['code'].is_unique)
check('التقييمات كلها من A إلى E (أو فاضية)', fp['grade'].dropna().isin(GRADES).all())
check('أغذية الرضع بدون تقييم', fp.loc[fp['product_group'] == 'Baby food', 'grade'].isna().all())
check('ما فيه سكر بالسالب', (fp['sugars'].dropna() >= 0).all())

# قواعد نظام البدائل
alt_groups = swaps.merge(pdf[['code', 'product_group']].rename(columns={'code': 'alt_code', 'product_group': 'alt_group'}),
                         on='alt_code')
check('كل بديل من نفس الفئة', (alt_groups['alt_group'] == alt_groups['product_group']).all())
check('كل بديل تقييمه أفضل',
      (swaps['alt_grade'].str.lower().map(GRADE_SCORE) > swaps['grade'].str.lower().map(GRADE_SCORE)).all())
check('كل بديل سكره أقل بـ 30% على الأقل', (swaps['alt_sugar_g'] <= swaps['sugar_g'] * 0.7).all())
check('ما فيه بديل يتكرر أكثر من 5 مرات', swaps['alt_name'].str.lower().str.strip().value_counts().max() <= 5)
check('تشابه الوسوم 0.2 أو أكثر لكل بديل', (swaps['tag_similarity'] >= 0.2).all())

assert all(ok for _, ok in results), 'فيه اختبار فشل!'
print('\n🎉 كل الاختبارات نجحت')

# الحفظ: يوصل هنا بس إذا كل الاختبارات نجحت
swaps.to_csv(f'{GOLD}/swap_recommendations.csv', index=False, encoding='utf-8-sig')
print('✅ تم الحفظ:', len(swaps), 'اقتراح لـ', swaps['code'].nunique(), 'منتج')
for f in sorted(os.listdir(REPORTS)):
    print(f'reports/{f}', round(os.path.getsize(f'{REPORTS}/{f}') / 1e3, 1), 'KB')

## ✅ خلصنا!
**المسار كامل (Notebook 1 + 2):**
Ingestion ← Data Quality ← Cleaning ← Classification + Validation ← Outlier Handling ← **SQL Analytics** ← **Recommendation Rules** ← Tests ← CSV

**المهارات اللي يغطيها المشروع:** PySpark، Spark SQL، Window Functions، تصميم Medallion (Bronze/Silver/Gold)، Data Quality، اختبارات البيانات، Pandas، ونظام توصيات مبني على قواعد.